<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c40-header.png" alt="Nextia Learning · Experiments and Causal Impact: Did Our Change Work?" width="100%">

# Solution: Did the mailing work, and for whom?

**[Did the mailing work, and for whom?](https://learning.nextia-ai.com/courses/experiments/m09/did-the-mailing-work-and-for-whom/)** · Experiments and Causal Impact: Did Our Change Work? · Case study 1 of 3 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will** analyse a real randomised mailing to 180,002 households, from the checks to the decision: check the randomisation, estimate each letter's effect with intervals clustered by household, compare the letters, look for whom the best letter works, and compare five ways to choose the households under a budget, with an uplift model among them.

| | |
|---|---|
| **Time** | About 2.5 hours with the lesson page; the notebook runs in about one minute after the install |
| **Needs** | An internet connection for the setup cells. No account and no key. |
| **You should know** | The checks before the result, from [Check the test before the result](https://learning.nextia-ai.com/courses/experiments/m03/check-the-test-before-the-result/). Segments, from [Segments and different effects](https://learning.nextia-ai.com/courses/experiments/m03/segments-and-different-effects/). Uplift and Qini curves, from [Uplift models](https://learning.nextia-ai.com/courses/experiments/m07/uplift-models/) and [Evaluate and use uplift](https://learning.nextia-ai.com/courses/experiments/m07/evaluate-and-use-uplift/). |
| **Data** | Gerber, Green and Larimer, *Social Pressure and Voter Turnout* (Yale ISPS Data Archive), CC0 1.0: a real mailing test before a 2006 US primary election, one row per registered voter |
| **Tested** | Python 3.12 with pandas 3.1.1, SciPy 1.15.3, statsmodels 0.15.0, scikit-learn 1.9.1 and matplotlib 3.11.2, in Jupyter on a laptop CPU (Apple M4, 16 GB), from a fresh start (*Restart and run all*) |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/experiments/m09/did-the-mailing-work-and-for-whom/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C40/M09-L01-did-the-mailing-work-and-for-whom/did-the-mailing-work-and-for-whom-solution.ipynb).

## Setup: install the packages

Run the next cell first. It installs the versions that the course uses: pandas 3.1.1, SciPy 1.15.3 and statsmodels 0.15.0, matplotlib 3.11.2 for the charts, and scikit-learn 1.9.1 for the models. It needs an internet connection and takes about a minute the first time.

You should see one line per package, with these versions. If you see other versions, restart the kernel (in Colab: **Runtime › Restart session**; in Kaggle: **Run › Restart & clear cell outputs**), then run this cell again. Restarting is needed when an older version was already loaded.

In [ ]:
%pip install -q --disable-pip-version-check pandas==3.1.1 "scipy==1.15.3; python_version < '3.14'" "scipy==1.18.1; python_version >= '3.14'" statsmodels==0.15.0 matplotlib==3.11.2 scikit-learn==1.9.1
import importlib.metadata as _md
for _name in ["pandas", "scipy", "statsmodels", "matplotlib", "scikit-learn"]:
    print(_name, _md.version(_name))

## Setup: get the data

Run the next cell. It downloads one file into a new folder `mailing-test` and checks its checksum:

- `social_pressure.tab` (20 MB): one row per person in a real mailing test, from the Yale ISPS Data Archive.

It needs an internet connection and no account, and takes about 10 seconds. You should see `Ready: social_pressure.tab (344084 rows).` If you run the cell again, it uses the file that is already there.

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings.
- `wrong checksum`: delete the folder `mailing-test` (in Colab: the **Files** panel on the left), then run the cell again. If it happens again, the source file has changed: tell us (the link is at the end of the notebook).

**Source and licence.** Gerber, Alan S.; Green, Donald P.; Larimer, Christopher W. *Social Pressure and Voter Turnout: Evidence from a Large-Scale Field Experiment.* Yale ISPS Data Archive, doi:[10.60600/YU/CGMWNW](https://doi.org/10.60600/YU/CGMWNW). Dedicated to the public domain under [CC0 1.0](https://creativecommons.org/publicdomain/zero/1.0/). Paper: *American Political Science Review* 102(1), 2008, 33–48. The notebook reads the original file and does not change it.

In [ ]:
# Setup: download the data into mailing-test/ and check it.
import hashlib, os, shutil, subprocess, urllib.request
from pathlib import Path

URL = "https://dataverse.yale.edu/api/access/datafile/42238"
SHA256 = "05807bcbedaacf02ec6e6838e1b9c9b2fc99ad9d8c63f80c162991970044c6ce"
HERE = Path.cwd()
if Path(".nextia-notebook").exists():  # the cell ran before: start from the parent folder
    os.chdir("..")
    HERE = Path.cwd()
PROJECT = Path("mailing-test").resolve()
PROJECT.mkdir(exist_ok=True)
(PROJECT / ".nextia-notebook").touch()
path = PROJECT / "social_pressure.tab"

def sha256(p):
    return hashlib.sha256(p.read_bytes()).hexdigest()

def download(url, p):
    try:
        urllib.request.urlretrieve(url, p)
    except OSError:
        # Python without certificates (python.org on macOS): try curl.
        if subprocess.run(["curl", "-fsSL", "-o", str(p), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

if not (path.exists() and sha256(path) == SHA256):
    if (HERE / path.name).exists():  # a copy next to the notebook
        shutil.copy(HERE / path.name, path)
    else:
        download(URL, path)
    if sha256(path) != SHA256:
        raise SystemExit("social_pressure.tab has the wrong checksum. Delete the folder mailing-test and run again.")
os.chdir(PROJECT)
print(f"Ready: social_pressure.tab ({sum(1 for _ in open(path)) - 1} rows).")

## Setup: helpers

The next cell loads pandas as `pd`, NumPy as `np`, SciPy's `stats` and statsmodels' formula interface as `smf`, and defines the check helpers. They tell you if your answer is right, without showing the answer:

- `expect(what, yours, expected)` compares one of your numbers with the expected one, and says "too high" or "too low".
- `expect_hash(what, yours, expected)` compares any result with a hidden fingerprint of the expected one.

Run the cell. It prints nothing.

In [ ]:
# Helpers: checks that do not show the answer, and the simulator's truth.
import hashlib, json, warnings
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from scipy import stats
pd.set_option("display.precision", 4)
pd.set_option("display.width", 140)
warnings.filterwarnings("ignore")

def fingerprint(value):
    def norm(v):
        if hasattr(v, "to_dict") and hasattr(v, "columns"):
            return {"columns": [str(c) for c in v.columns], "rows": [norm(list(r)) for r in v.itertuples(index=False)]}
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple)):
            return [norm(x) for x in v]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 4)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
        return
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. The data

One row is one registered voter. The study sent one letter per **household** (people at the same address), or no letter. The outcome `voted` (1 or 0) comes from public records after the election.

The columns that we use:

| Column | What it means |
|---|---|
| `treatment` | the letter: 0 no letter, 1 Hawthorne, 2 Civic duty, 3 Neighbors, 4 Self |
| `voted` | 1 if the person voted in the primary election of August 2006 (the outcome) |
| `g2000`, `g2002`, `p2000`, `p2002`, `p2004` | 1 if the person voted in that earlier general (g) or primary (p) election |
| `g2004` | the 2004 general election: everyone in the file voted in it |
| `hh_id`, `hh_size` | the household, and the number of people from it in the file |
| `yob` | year of birth |

We do not use `sex`. The lesson page explains why.

In [ ]:
ARMS = {0: "No letter", 1: "Hawthorne", 2: "Civic duty", 3: "Neighbors", 4: "Self"}
LETTERS = ["Civic duty", "Hawthorne", "Self", "Neighbors"]
PAST = ["g2000", "g2002", "p2000", "p2002", "p2004"]  # the five earlier elections
FEATURES = PAST + ["hh_size", "age"]
Z = stats.norm.ppf(0.975)

raw = pd.read_csv("social_pressure.tab", sep="\t")
people = raw.assign(letter=raw["treatment"].map(ARMS), past_votes=raw[PAST].sum(axis=1), age=2006 - raw["yob"])
print("rows, columns:", raw.shape, "| households:", people["hh_id"].nunique(), "| missing:", int(raw.isna().sum().sum()))
print("everyone voted in g2004:", bool((raw["g2004"] == 1).all()))

> **Check.** You should see `rows, columns: (344084, 16) | households: 180002 | missing: 0` and `everyone voted in g2004: True`.

## 2. The plan before the result

We write the plan **before** we look at `voted`:

1. **Checks:** the number of households per group against the design (5 : 1 : 1 : 1 : 1), balance of the facts known before the letters, a placebo outcome, and A/A splits of the no-letter group.
2. **The effect of each letter** against no letter: the difference in the share who voted, with a 95% interval clustered by household. Four comparisons.
3. **For whom:** six planned segments, by the number of the five earlier elections the person voted in (0 to 5).
4. **Whom to mail:** a budget of letters for one household in five. Five ways to rank households, compared by one measure: extra voters per 1,000 letters to the top 20%.

## 3. Check the randomisation

First, the households per group. The design put 5 of every 9 households in the no-letter group, and 1 of 9 in each letter group.

In [ ]:
house = people.groupby("hh_id").agg(letter=("letter", "first"), size=("voted", "size"))
counts = house["letter"].value_counts().reindex(list(ARMS.values()))
planned = np.array([5, 1, 1, 1, 1]) / 9
chi = stats.chisquare(counts.to_numpy(), planned * counts.sum())
print(counts.to_dict())
print(f"chi-square {chi.statistic:.2f}, p = {chi.pvalue:.3f}")

> **Check.** You should see `chi-square 0.00, p = 1.000`. No sample ratio mismatch.

Next, **balance**: the facts known before the letters should be alike in every group. The standardised difference is the gap in means divided by the spread; under 0.1 is the usual rule of thumb.

In [ ]:
ctrl = people[people["letter"] == "No letter"]
rows = {}
for col in FEATURES:
    rows[col] = {arm: (people.loc[people["letter"] == arm, col].mean() - ctrl[col].mean())
                 / np.sqrt((people.loc[people["letter"] == arm, col].var() + ctrl[col].var()) / 2) for arm in LETTERS}
balance = pd.DataFrame(rows).T
print(balance.round(4))
print(f"largest standardised difference: {balance.abs().max().max():.4f}")

> **Check.** You should see `largest standardised difference: 0.0129`.

Now the main tool of this case study. The coin was flipped per **household**, and people at one address often vote together. So we compute the interval from household totals: an interval **clustered by household**. The function also gives the naive interval that treats every person as independent, so that we can compare.

In [ ]:
def by_household(df, outcome):
    """Household totals of an outcome: the household is the unit of randomisation."""
    return df.groupby("hh_id").agg(arm=("letter", "first"), n=(outcome, "size"), y=(outcome, "sum"))

def effect(g, arm, base="No letter"):
    """Letter minus no letter, in the share: the difference, a 95% interval clustered by household, and the naive one."""
    parts = {}
    for name in (base, arm):
        h = g[g["arm"] == name]
        n, y = h["n"].to_numpy(float), h["y"].to_numpy(float)
        r, G = y.sum() / n.sum(), len(h)
        parts[name] = (r, G / (G - 1) * np.sum((y - r * n) ** 2) / n.sum() ** 2, r * (1 - r) / (n.sum() - 1))
    d = parts[arm][0] - parts[base][0]
    se, se_naive = np.sqrt(parts[arm][1] + parts[base][1]), np.sqrt(parts[arm][2] + parts[base][2])
    return {"rate_base": parts[base][0], "rate_arm": parts[arm][0], "diff": d, "low": d - Z * se, "high": d + Z * se,
            "low_naive": d - Z * se_naive, "high_naive": d + Z * se_naive, "p": 2 * stats.norm.sf(abs(d / se)),
            "se": se, "se_naive": se_naive}

A **placebo outcome** is something that the letters cannot change: here, voting in the 2004 primary, two years before. Its "effect" must be about zero.

> **Predict.** Will all four intervals contain zero?

In [ ]:
gp = by_household(people, "p2004")
for arm in LETTERS:
    e = effect(gp, arm)
    print(f"{arm:<11} {e['diff']:+.4f}  95% {e['low']:+.4f} to {e['high']:+.4f}  p = {e['p']:.2f}")

> **Check.** You should see four intervals that contain zero; the Neighbors line is `Neighbors   +0.0063  95% -0.0004 to +0.0130  p = 0.07`.

Last check: **A/A splits**. We split the no-letter households at random into two fake groups (20,000 and the rest) and test the difference, 1,000 times. Nothing differs, so a good method calls about 5% of the splits "significant".

In [ ]:
cg = by_household(ctrl, "voted")
n_c, y_c = cg["n"].to_numpy(float), cg["y"].to_numpy(float)

def aa_false_winners(n_fake=20000, reps=1000, seed=2006):
    """Share of random splits of the no-letter households called significant at 5%: clustered, then naive."""
    rng = np.random.default_rng(seed)
    fp_cl = fp_nv = 0
    for _ in range(reps):
        fake = np.zeros(len(cg), bool)
        fake[rng.choice(len(cg), n_fake, replace=False)] = True
        v_cl, v_nv, rates = [], [], []
        for m in (fake, ~fake):
            n, y = n_c[m], y_c[m]
            r, G = y.sum() / n.sum(), m.sum()
            v_cl.append(G / (G - 1) * np.sum((y - r * n) ** 2) / n.sum() ** 2)
            v_nv.append(r * (1 - r) / (n.sum() - 1))
            rates.append(r)
        d = rates[0] - rates[1]
        fp_cl += abs(d) / np.sqrt(sum(v_cl)) > Z
        fp_nv += abs(d) / np.sqrt(sum(v_nv)) > Z
    return fp_cl / reps, fp_nv / reps

clustered, naive = aa_false_winners()
print(f"'significant' at 5%: clustered {clustered:.1%}, naive (people independent) {naive:.1%}")

> **Check.** You should see `'significant' at 5%: clustered 5.8%, naive (people independent) 12.7%`. The naive interval finds a "winner" in about one split in eight: it is too narrow.

## 4. The effect of each letter

Now, and only now, we read `voted`.

In [ ]:
gv = by_household(people, "voted")
effects = {}
for arm in LETTERS:
    e = effects[arm] = effect(gv, arm)
    print(f"{arm:<11} {e['rate_base']:.2%} -> {e['rate_arm']:.2%}: {e['diff']:+.4f}  clustered 95% {e['low']:+.4f} to "
          f"{e['high']:+.4f} | naive {e['low_naive']:+.4f} to {e['high_naive']:+.4f} | width ratio {e['se'] / e['se_naive']:.2f}")

> **Check.** You should see four lines; the last one is `Neighbors   29.66% -> 37.79%: +0.0813  clustered 95% +0.0747 to +0.0879 | naive +0.0760 to +0.0866 | width ratio 1.25`.

The same differences from a regression, with statsmodels' clustered errors, and then with the five past votes added (regression adjustment, as CUPED: a little narrower). This takes about 20 seconds.

In [ ]:
from threadpoolctl import threadpool_limits

data = people.assign(letter=pd.Categorical(people["letter"], categories=list(ARMS.values())))
with threadpool_limits(1):
    plain = smf.ols("voted ~ C(letter)", data=data).fit(cov_type="cluster", cov_kwds={"groups": data["hh_id"]})
    adjusted = smf.ols("voted ~ C(letter) + " + " + ".join(PAST), data=data).fit(
        cov_type="cluster", cov_kwds={"groups": data["hh_id"]})
for arm in LETTERS:
    k = f"C(letter)[T.{arm}]"
    (lo, hi), (alo, ahi) = plain.conf_int().loc[k], adjusted.conf_int().loc[k]
    print(f"{arm:<11} regression {plain.params[k]:+.4f} ({lo:+.4f} to {hi:+.4f}); "
          f"with past votes {adjusted.params[k]:+.4f} ({alo:+.4f} to {ahi:+.4f})")

> **Check.** The first column matches the section above. The Neighbors line is `Neighbors   regression +0.0813 (+0.0747 to +0.0879); with past votes +0.0807 (+0.0744 to +0.0870)`.

### Compare the letters

Is Neighbors better than Self? Test the **difference** between the two letters, not two p-values.

In [ ]:
cov = plain.cov_params()
for a, b in (("Neighbors", "Self"), ("Self", "Hawthorne"), ("Hawthorne", "Civic duty")):
    ka, kb = f"C(letter)[T.{a}]", f"C(letter)[T.{b}]"
    d = plain.params[ka] - plain.params[kb]
    se = np.sqrt(cov.loc[ka, ka] + cov.loc[kb, kb] - 2 * cov.loc[ka, kb])
    print(f"{a} - {b}: {d:+.4f} (95% {d - Z * se:+.4f} to {d + Z * se:+.4f})")

> **Check.** You should see `Neighbors - Self: +0.0328 (95% +0.0243 to +0.0413)` on the first line.

The chart shows each letter's effect with both intervals.

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(7, 3))
for i, arm in enumerate(LETTERS):
    e = effects[arm]
    ax.plot([100 * e["low"], 100 * e["high"]], [i + 0.1] * 2, color="tab:blue", lw=3, label="clustered" if i == 0 else None)
    ax.plot([100 * e["low_naive"], 100 * e["high_naive"]], [i - 0.1] * 2, color="tab:gray", lw=3, label="naive" if i == 0 else None)
    ax.plot(100 * e["diff"], i, "ko")
ax.axvline(0, color="black", ls="--", lw=1)
ax.set_yticks(range(4), LETTERS)
ax.set_xlabel("Effect on turnout (percentage points)")
ax.legend(loc="lower right")
plt.tight_layout()
plt.show()

## 5. For whom: six planned segments

The Neighbors letter, by the number of the five earlier elections the person voted in.

In [ ]:
segments = {}
for k in range(6):
    sub = people[people["past_votes"] == k]
    e = segments[k] = effect(by_household(sub, "voted"), "Neighbors")
    print(f"{k} of 5: {len(sub):>6} people ({len(sub) / len(people):.1%}); no letter {e['rate_base']:.1%}; "
          f"effect {e['diff']:+.4f} (95% {e['low']:+.4f} to {e['high']:+.4f})")

> **Check.** You should see six lines. People who voted in 3 or 4 of 5 respond most: `4 of 5:  65852 people (19.1%); no letter 41.9%; effect +0.0987 (95% +0.0848 to +0.1126)` for 4 of 5.

> **Your turn.** Estimate the **Self** letter's effect among people who voted in 3 of the 5 earlier elections. Use `effect()` and `by_household()`. Put the difference in `self_3`, rounded to 4 decimals.

In [ ]:
sub = people[people["past_votes"] == 3]
self_3 = round(float(effect(by_household(sub, "voted"), "Self")["diff"]), 4)
print(self_3)

In [ ]:
expect('self_3', self_3, 0.0556, tolerance=0.0005)

## 6. Whom to mail: five lists under one measure

The budget: letters for one household in five. We compare five ways to rank the households for the Neighbors letter:

- `size`: the biggest households first (one letter reaches every voter in the household);
- `segment`: the planned segments' effects, learnt from the training households;
- `response`: a "who will vote" model, trained on the households that got the letter;
- `t` and `x`: the T-learner and the X-learner from the uplift lessons.

To be fair, every household gets its score from models that **never saw it**. We cut the households into 4 folds, train on 3 and score the fourth, four times (**cross-fitting**). This takes about a minute.

In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier, HistGradientBoostingRegressor
from sklearn.linear_model import LogisticRegression

two = people[people["letter"].isin(["No letter", "Neighbors"])].copy()
two["t"] = (two["letter"] == "Neighbors").astype(int)
hh = np.sort(two["hh_id"].unique())
K = 4
fold_of = pd.Series(np.random.default_rng(0).integers(0, K, len(hh)), index=hh)
two["fold"] = two["hh_id"].map(fold_of)

def tree():
    return HistGradientBoostingRegressor(max_iter=150, learning_rate=0.05, max_leaf_nodes=15, min_samples_leaf=200,
                                         random_state=0)

def fit_models(train):
    """Every list's scoring rule, learnt on the training households only. Each returns a score per person."""
    X, y, t = train[FEATURES].to_numpy(float), train["voted"].to_numpy(float), train["t"].to_numpy(int)
    seg = train.groupby(["past_votes", "t"])["voted"].mean().unstack()
    seg_effect = (seg[1] - seg[0]).to_dict()
    resp = HistGradientBoostingClassifier(max_iter=150, learning_rate=0.05, max_leaf_nodes=15, min_samples_leaf=200,
                                          random_state=0).fit(X[t == 1], y[t == 1])
    ma, mb = tree().fit(X[t == 0], y[t == 0]), tree().fit(X[t == 1], y[t == 1])
    ea = tree().fit(X[t == 0], mb.predict(X[t == 0]) - y[t == 0])
    eb = tree().fit(X[t == 1], y[t == 1] - ma.predict(X[t == 1]))
    prop = LogisticRegression(max_iter=1000).fit(X, t)

    def x_score(Xd):
        g = prop.predict_proba(Xd)[:, 1]
        return g * ea.predict(Xd) + (1 - g) * eb.predict(Xd)
    return {
        "size": lambda d: np.ones(len(d)),
        "segment": lambda d: d["past_votes"].map(seg_effect).to_numpy(float),
        "response": lambda d: resp.predict_proba(d[FEATURES].to_numpy(float))[:, 1],
        "t": lambda d: mb.predict(d[FEATURES].to_numpy(float)) - ma.predict(d[FEATURES].to_numpy(float)),
        "x": lambda d: x_score(d[FEATURES].to_numpy(float)),
    }

NAMES = ["size", "segment", "response", "t", "x"]
scores = {n: np.zeros(len(two)) for n in NAMES}
with threadpool_limits(1):
    for k in range(K):
        m = fit_models(two[two["fold"] != k])
        held = (two["fold"] == k).to_numpy()
        for n in NAMES:
            scores[n][held] = m[n](two[held])
for n in NAMES:
    two[f"s_{n}"] = scores[n]
print(f"{len(two):,} people in {len(hh):,} households, scored out of fold")

> **Check.** You should see `229,444 people in 119,999 households, scored out of fold`.

One letter goes to a household, so a household's score is the **sum** of its people's scores. Then the course's uplift curve and Qini coefficient (the same code as the project's `uplift.py`), and our one measure: extra voters per 1,000 letters to the top 20% of households.

In [ ]:
H = (two.groupby("hh_id").agg(y=("voted", "sum"), t=("t", "first"), n=("voted", "size"),
                              **{f"s_{n}": (f"s_{n}", "sum") for n in NAMES}))
H = H.iloc[np.random.default_rng(7).permutation(len(H))]  # a fixed random order: ties are broken at random

def uplift_curve(score, y, t, points=20):
    """Sorted by score, at each cut: (mean of the letter group − mean of the no-letter group) × households above it."""
    order = np.argsort(-score, kind="stable")
    y, t = y[order], t[order]
    n = len(y)
    shares, gains = [0.0], [0.0]
    for k in np.linspace(n / points, n, points).astype(int):
        yt, yc, nt, nc = y[:k][t[:k] == 1].sum(), y[:k][t[:k] == 0].sum(), (t[:k] == 1).sum(), (t[:k] == 0).sum()
        shares.append(k / n)
        gains.append(float((yt / max(nt, 1) - yc / max(nc, 1)) * k))
    return np.array(shares), np.array(gains)

def qini(shares, gains):
    """The area between the curve and the straight line of a random list."""
    return float(np.trapezoid(gains - gains[-1] * shares, shares))

def top_gain(score, y, t, share=0.2):
    """Extra voters per 1,000 letters to the top share of households, and its standard error."""
    order = np.argsort(-score, kind="stable")[: int(round(share * len(y)))]
    yy, tt = y[order], t[order]
    a, b = yy[tt == 1], yy[tt == 0]
    return 1000 * (a.mean() - b.mean()), 1000 * np.sqrt(a.var(ddof=1) / len(a) + b.var(ddof=1) / len(b))

y_h, t_h = H["y"].to_numpy(float), H["t"].to_numpy(int)
curves, top20 = {}, {}
for n in NAMES:
    s = H[f"s_{n}"].to_numpy()
    curves[n] = uplift_curve(s, y_h, t_h)
    d, se = top20[n] = top_gain(s, y_h, t_h)
    print(f"{n:<9} top 20%: {d:+.1f} extra voters per 1,000 letters (95% {d - Z * se:+.1f} to {d + Z * se:+.1f}); "
          f"Qini {qini(*curves[n]):.1f}")
d_all, se_all = top_gain(np.zeros(len(H)), y_h, t_h, 1.0)
print(f"everyone: {d_all:+.1f} per 1,000 letters (95% {d_all - Z * se_all:+.1f} to {d_all + Z * se_all:+.1f})")

> **Check.** You should see `x         top 20%: +231.1 extra voters per 1,000 letters (95% +197.3 to +264.9); Qini 1830.4` on the `x` line, and `everyone: +154.6 per 1,000 letters (95% +141.6 to +167.6)` for everyone.

The intervals overlap. Are the lists really different? A **paired bootstrap** resamples the households and compares two lists on the **same** resample, so that the shared noise cancels. This takes about 30 seconds.

In [ ]:
rng_p = np.random.default_rng(5)
boots = {n: [] for n in NAMES}
for _ in range(200):
    i = rng_p.integers(0, len(H), len(H))
    for n in NAMES:
        boots[n].append(top_gain(H[f"s_{n}"].to_numpy()[i], y_h[i], t_h[i])[0])
best = max(NAMES, key=lambda n: top20[n][0])
print("the best list by the one measure:", best)
for n in NAMES:
    if n != best:
        lo, hi = np.percentile(np.array(boots[best]) - np.array(boots[n]), [2.5, 97.5])
        print(f"{best} - {n:<9} {top20[best][0] - top20[n][0]:+.1f} (95% {lo:+.1f} to {hi:+.1f})")

> **Check.** You should see `x - size      +45.5 (95% +6.1 to +78.3)` on the line against `size`.

The curves, against a random list:

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
for n in NAMES:
    s, g = curves[n]
    ax.plot(100 * s, g, label=n)
ax.plot([0, 100], [0, curves["x"][1][-1]], "k--", label="random list")
ax.set_xlabel("Share of households mailed (%)")
ax.set_ylabel("Extra voters")
ax.legend()
plt.tight_layout()
plt.show()

### A placebo for the model

A real test has no "true" uplift to compare with. So we make a test where the truth is known to be zero: a **fake letter** inside the no-letter group, one household in six. A model that "finds" uplift here finds noise.

In [ ]:
c_only = two[two["t"] == 0].copy()
c_hh = np.sort(c_only["hh_id"].unique())
fake = pd.Series(np.random.default_rng(3).random(len(c_hh)) < 1 / 6, index=c_hh)
c_only["t"] = c_only["hh_id"].map(fake).astype(int)
ps = np.zeros(len(c_only))
with threadpool_limits(1):
    for k in range(K):
        m = fit_models(c_only[c_only["fold"] != k])
        held = (c_only["fold"] == k).to_numpy()
        ps[held] = m["x"](c_only[held])
Hp = c_only.assign(s=ps).groupby("hh_id").agg(y=("voted", "sum"), t=("t", "first"), s=("s", "sum"))
Hp = Hp.iloc[np.random.default_rng(7).permutation(len(Hp))]
dp, sep = top_gain(Hp["s"].to_numpy(), Hp["y"].to_numpy(float), Hp["t"].to_numpy())
print(f"placebo top 20%: {dp:+.1f} per 1,000 (95% {dp - Z * sep:+.1f} to {dp + Z * sep:+.1f})")

> **Check.** You should see `placebo top 20%: +6.1 per 1,000 (95% -26.7 to +38.9)`: zero is inside the interval.

## 7. The chosen list on new input

The households that got another letter stand in for next year's list. We use only their facts known before the mailing (no outcome), score them with the chosen model trained on all the households of the Neighbors test, and choose the top 20%.

In [ ]:
with threadpool_limits(1):
    final = fit_models(two)
new = people[~people["letter"].isin(["No letter", "Neighbors"])].copy()
new["score"] = final[best](new)
plan = (new.groupby("hh_id").agg(score=("score", "sum"), people=("voted", "size"), past=("past_votes", "mean"))
        .pipe(lambda d: d.iloc[np.random.default_rng(7).permutation(len(d))]))
chosen = plan.sort_values("score", ascending=False, kind="stable").head(int(round(0.2 * len(plan))))
print(f"{len(plan):,} households; chosen {len(chosen):,} letters to {int(chosen['people'].sum()):,} people; "
      f"expected extra voters {chosen['score'].sum():.0f} ({1000 * chosen['score'].mean():.1f} per 1,000 letters)")

> **Check.** You should see `60,003 households; chosen 12,001 letters to 30,095 people; expected extra voters 2696 (224.7 per 1,000 letters)`.

> **Your turn.** A letter costs 0.50 dollars to print and post (our assumption). Measured above, the chosen list's top 20% gave `top20[best][0]` extra voters per 1,000 letters. What does one extra voter cost with that list? Put it in `cost_per_voter`, in dollars, rounded to 2 decimals.

In [ ]:
cost_per_voter = round(1000 * 0.50 / top20[best][0], 2)
print(cost_per_voter)

In [ ]:
expect('cost_per_voter', cost_per_voter, 2.16, tolerance=0.005)

> **Your turn: change one thing.** Run the A/A splits again with a fake group of **50,000** households instead of 20,000 (`aa_false_winners(n_fake=50000)`). Predict first: will the naive method's false winners go up or down? Put the naive share in `naive_50k`, rounded to 3 decimals.

In [ ]:
naive_50k = round(float(aa_false_winners(n_fake=50000)[1]), 3)
print(naive_50k)

In [ ]:
expect('naive_50k', naive_50k, 0.138, tolerance=0.0005)

## Next

The [lesson page](https://learning.nextia-ai.com/courses/experiments/m09/did-the-mailing-work-and-for-whom/) explains each step, gives the recommendation and its limits, and has the knowledge checks that count toward your certificate. Then make it a portfolio piece, as the page shows.

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Real tests at an online shop](https://learning.nextia-ai.com/courses/experiments/m09/real-tests-at-an-online-shop/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/experiments/m09/did-the-mailing-work-and-for-whom/)

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The data is real and public (CC0 1.0): a 2006 US election study. It has no names or addresses. Do not try to identify anyone.